# FLP - příprava na cvičení č. 5 (Rust 2) - až 2b
Toto je notebook s přípravou na výše uvedené cvičení.

Pokyny:
- Nastudujte si informace uvedené v tomto notebooku.
- Vyřešte příklady uvedené v sekcích **PŘÍKLAD**, např. doplněním zdrojového kódu do vyznačených částí / opravou kódu - dle pokynů.
- Do kódu/textu mimo příklady **nezasahujte**, žádné další buňky/bloky kódu **nepřidávejte**.
- Notebook s vyřešenými příklady nahrajte do Moodlu v sekci **Odevzdání domácích příprav**.

Na cvičení se bude **předpokládat** znalost zde uvedené problematiky, stejně jako znalost látky z dosud proběhlých přednášek!

Je **PŘÍSNĚ ZAKÁZÁNO** tento soubor poskytovat jiným osobám, nebo s nimi sdílet své řešení!

### Poznámka k využití generativní AI při řešení
Využití generativní AI zakázáno není, avšak **silně doporučujeme** se nejprve nad příkladem zamyslet **samostatně** a konzultaci s generativní AI brát až jako poslední možnost. Tento notebook je zde pro Vás, abyste se něco naučili, něco si vyzkoušeli a trochu se u toho "potrápili". Získané zkušenosti se vám budou hodit na **cvičeních**, při řešení **projektu** i u **zkoušky**. Necháte-li si "řešení vygenerovat chatbotem", ochuzujete především sami sebe.

## Strukturované datové typy v Rustu
- **n-tice** (tuples) - pevný počet prvků, obecně různé typy (heterogenní)
- **pole** (arrays) - sekvence hodnot stejného typu, pevná velikost, souvislé uložení
- **řetězec** (`String`) - dynamická textová data v UTF-8
- **řez** (slice) - „pohled“ na souvislou sekvenci prvků uloženou jinde (jen reference)
- **struktura** (`struct`) - pojmenovaný typ obsahující dvojice *klíč* - *hodnota*
- **výčtové typy** (`enum`) - jedna z více variant, každá si může nést data
- **smart pointery a obaly** (např. `Box`) - struktury pro řízení vlastnictví a sdílení dat

### Kolekce:
- **vektor** (`Vec<T>`) - dynamické sekvence prvků stejného typu
- **oboustranná fronta** (`VecDeque<T>`) - efektivní přidávání/odebírání na obou koncích
- **spojový seznam** (`LinkedList<T>`) - klasický spojový seznam á la IAL, *dvousměrně* vázaný
- **hash množina** (`HashSet<T>`) - neseřazená množina unikátních prvků *stejného typu*, rychlé testy členství
- **B-stromová množina** (`BTreeSet<T>`) - *seřazená* množina unikátních prvků *stejného typu*, deterministické iterace, rozsahové dotazy nad klíči (od..do, menší/větší, ..)
- **binární halda** (`BinaryHeap<T>`) - implementace prioritní fronty, *max-heap* - nejvyšší prioritu má kořen
- **hash mapa** (`HashMap<K, V>`) - neseřazený slovník párů *klíč* - *hodnota*, rychlé vyhledávání
- **B-stromová mapa** (`BTreeMap<K, V>`) - *seřazený* slovník párů *klíč* - *hodnota*, deterministické iterace, rozsahové dotazy nad klíči (od..do, menší/větší, ..)




## N-tice (tuple)
- Pevná posloupnost hodnot obecně *různých* typů
- Indexujeme vždy přes **tečkovou notaci** (např. `x.1`). Hranaté závorky jsou pro pole/slices/vektory.

In [2]:
fn main() {
    let x = (42, true, 3.14);        // (i32, bool, f64) - odvozeno
    let y: (i32, bool) = (7, false); // explicitne
    
    // Pristup pres .0, .1, ...
    println!("{}", x.0); // 42
    println!("{}", y.1); // true
    
    // Rozklad na jednotlive prvky:
    let (a, b, c) = x;
    println!("{a} {b} {c}");
}
main();

42
false
42 true 3.14


### N-tice pro návrat více hodnot z funkce
Chceme-li, aby funkce vracela více než 1 hodnotu, často se používá n-tice.

In [3]:
fn divmod(a: i32, b: i32) -> (i32, i32) {
    (a / b, a % b)
}

fn main() {
    let (q, r) = divmod(17, 5);
    println!("q={q}, r={r}"); // q=3, r=2
}
main();

q=3, r=2


U rozsáhlejší dat to však nebývá úplně přehledné...

### **PŘÍKLAD 1:** Práce s n-ticemi
Doplňte chybějící kód u dvou funkcí, které pracují s dvojicemi celých čísel `i32` (resp. referencemi na ně):
- `get_first` - vrátí první prvek z dvojice
- `get_second` - vrátí druhý prvek z dvojice
- `swap` - prohodí prvky ve dvojici
- `max` - vrátí hodnotu většího z prvků
- Do funkce `main` a definice názvů/parametrů/návratových hodnot funkcí **nezasahujte**!

In [9]:
fn get_first(pair: &(i32, i32)) -> i32 {
    pair.0
}

fn get_second(pair: &(i32, i32)) -> i32 {
    pair.1
}

fn swap(pair: &mut (i32, i32)) {
    let tmp = pair.0;
    pair.0 = pair.1;
    pair.1 = tmp;
}

fn max(pair: &(i32, i32)) -> i32 {
    if pair.0 > pair.1 {
        pair.0
    } else {
        pair.1
    }
}

fn main() {
    let mut p = (10, 20);

    println!("first = {}", get_first(&p));
    assert_eq!(get_first(&(1, 2)), 1);

    println!("second = {}", get_second(&p));
    assert_eq!(get_second(&(1, 2)), 2);

    println!("max = {}", max(&p));
    assert_eq!(max(&(10, 20)), 20);

    swap(&mut p);
    println!("after swap = {:?}", p);
    assert_eq!(p, (20, 10));
}

main();

first = 10
second = 20
max = 20
after swap = (20, 10)


## Pole (array)
- *Homogenní* strukturovaný typ *pevné velikosti*.
- Typ: `[typ; velikost]`, typicky na stacku (není-li součástí něčeho na heapu)
- Inicializace: výčtem -- např. `a = [1, 2, 3];`
- Inicializace hodnotou/počtem -- např. `a = [1; 5];` (pole pěti jedniček)
- Přístup: `a[i]` (bounds-check, mimo rozsah → panic), bezpečně `a.get(i)` či `v.get_mut(i)`
- `a.first()`, `a.first_mut()`, `a.last()`, `a.last_mut()` - první/poslední prvek
- `a.len()` - počet prvků pole
- Iterace: `for x in a.iter()` nebo `for x in a.iter_mut()`
- (Pozn.: Iterace lze i `for x in a`, ale ve starších verzích Rustu nejde přes reference a může prvky "spotřebovat".)
- `a.sort()` / `a.sort_by(|a, b| ...)` - řazení / řazení s vlastní porovnávací funkcí

In [10]:
fn main() {
    let x = ['A', 'B', 'C'];
    println!("{:?}", x);
    
    let mut a: [i32; 4] = [1, 2, 3, 4];
    let zeros = [0u8; 3];
    
    a[0] += 10; // indexovani (pozor na rozsah!)
    for x in a.iter() { println!("{x}"); }
    for x in a.iter_mut() { *x += 1; }
    
    // :? - debug print
    println!("{:?} {:?}", a, zeros);  // [12, 3, 4, 5] [0, 0, 0]
}
main();

['A', 'B', 'C']
11
2
3
4
[12, 3, 4, 5] [0, 0, 0]


### **PŘÍKLAD 2:** Práce s poli
Doplňte chybějící kód do funkcí pro práci s polem:
- `arr_count_odd` - vrátí **počet** lichých čísel v poli
- `arr_sum_even` - vrátí **součet** sudých čísel v poli
- `arr_sort` - seřadí pole **vzestupně**
- `arr_sort_desc` - seřadí pole **sestupně**
- Do funkce `main` a definice názvů/parametrů/návratových hodnot funkcí **nezasahujte**!

In [11]:
fn arr_count_odd(a: &[i32]) -> usize {
    a.iter().filter(|&&x| x % 2 != 0).count()
}

fn arr_sum_even(a: &[i32]) -> i32 {
    a.iter().filter(|&&x| x % 2 == 0).sum()
}

fn arr_sort(a: &mut [i32]) {
    a.sort();
}

fn arr_sort_desc(a: &mut [i32]) {
    a.sort_by(|a, b| b.cmp(a));
}

fn main() {
    let a5 = [5, 1, 4, 2, 3];
    let a3 = [7, 2, 9];

    println!("count_odd(a5) = {}", arr_count_odd(&a5));
    assert_eq!(arr_count_odd(&a5), 3);

    println!("sum_even(a5) = {}", arr_sum_even(&a5));
    assert_eq!(arr_sum_even(&a5), 6);

    println!("count_odd(a3) = {}", arr_count_odd(&a3));
    assert_eq!(arr_count_odd(&a3), 2);

    println!("sum_even(a3) = {}", arr_sum_even(&a3));
    assert_eq!(arr_sum_even(&a3), 2);

    let mut b = a5;
    arr_sort(&mut b);
    println!("sorted(b) = {:?}", b);
    assert_eq!(b, [1, 2, 3, 4, 5]);

    let mut c = a5;
    arr_sort_desc(&mut c);
    println!("sorted_desc(c) = {:?}", c);
    assert_eq!(c, [5, 4, 3, 2, 1]);

    // puvodni pole zustava beze zmeny
    println!("original a5 = {:?}", a5);
}

main();

count_odd(a5) = 3
sum_even(a5) = 6
count_odd(a3) = 2
sum_even(a3) = 2
sorted(b) = [1, 2, 3, 4, 5]
sorted_desc(c) = [5, 4, 3, 2, 1]
original a5 = [5, 1, 4, 2, 3]


## Vektor (Vec\<T\>)
- Dynamické pole
- Vytvoření: `Vec::new()` nebo makro `vec![...]`
- Přidání/odebrání: `push(x)`, `pop()`, `insert(i, x)`, `remove(i)`, `append(v)`, ...
- Přístup k prvku: `v[i]` (přístup za hranice → panic), bezpečně `v.get(i)` či `v.get_mut(i)`
- `v.first()`, `v.first_mut()`, `v.last()`, `v.last_mut()` - první/poslední prvek
- Iterace s přesunem: `for x in v` - přesune prvky **pryč z vektoru**
- Iterace přes reference: `for x in &v`, změny: `for x in &mut v`


In [12]:
fn main() {
    let mut v: Vec<i32> = vec![10, 20, 30];

    v.push(40);
    println!("len={}", v.len());          // len=4
    println!("v[1]={}", v[1]);            // v[1]=20
    println!("get(10)={:?}", v.get(10));  // get(10)=None
    for x in &v { println!("{x}"); }      // 10 20 30 40
    for x in &mut v { *x += 1; }
}
main();

len=4
v[1]=20
get(10)=None
10
20
30
40


## Uložení vektoru v paměti

- Na **stacku** je *handle* - délka, kapacita, ukazatel.
- Na **heapu** jsou data.
```
   Stack (zásobník)                 Heap (halda)
   +---------------------+          +---------------------+
   | +-----------------+ |          |                     |
   | | VEC STRUCT      | |          |                     |
   | | ptr ------------+-+----------+-> [1, 2, 3, 4, 5]   |
   | | len       = 5   | |          |                     |
   | | capacity  = 7   | |          |                     |
   | +-----------------+ |          |                     |
   |                     |          |                     |
   +---------------------+          +---------------------+
```
**POZOR:** Toto neplatí, pokud je vektor součástí něčeho na heapu - např. vektor uvnitř vektoru.

### Vektor a kapacita
- Vektor funguje podobně jako **nafukovací pole** (viz IAL)
- `Vec::new()` vytvoří prázdný vektor: `len = 0`, typicky `capacity = 0` (zatím bez alokace na heapu).
- První `push/extend` obvykle provede alokaci: `capacity` se zvětší na > 0 (přesná kapacita *není garantovaná*).
- Když `len == capacity` a přidáváme další prvek, dojde k **realokaci**:
  - alokuje se nový (větší) blok paměti,
  - prvky se přesunou/zkopírují do nového bloku,
  - starý blok se uvolní,
  - v handle se aktualizuje `ptr` a `capacity`.
- Kapacita roste geometricky (většinou zhruba násobením ~2), ale přesná čísla *nejsou garantovaná*.
- `with_capacity(n)`/`reserve(n)` umí **realokacím předejít** tím, že místo **předem zarezervují**.
- **POZOR:** Po realokaci mohou být **neplatné** dříve získané reference/pointery do prvků.
- Proto **nelze** volat `reserve(n)`, **existuje-li** reference na původní vektor, nebo jeho prvky (vector slice)!


In [13]:
fn main() {
    let mut v: Vec<i32> = Vec::new();
    println!("len={}, cap={}", v.len(), v.capacity()); // 0, 0

    v.push(1);
    println!("len={}, cap={}", v.len(), v.capacity()); // cap uz bude > 0 (kolik presne NENI GARANTOVANO)
}
main();

len=0, cap=0
len=1, cap=4


## Užitečné operace s `Vec<T>`
- `Vec::with_capacity(n)` - předalokování paměti (pri inicializaci)
- `reserve (additional_n)` - doalokace další paměti
- `extend(pole/rozsah/jinyvektor)` - přidá více prvků najednou 
- `len()`, `capacity()`, `is_empty()`
- `clear()` - vymaže všechny prvky
- `truncate(n)` - odstraní prvky od indexu `n` dále
- `retain(|x| ...)` - ponechá jen prvky splňující podmínku

### Řazení a vyhledávání
- `sort()`, `sort_by(|a, b| ...)` - řazení, řazení dle vlastní porovnávací funkce
- `binary_search(&x)` - binární vyhledávání na seřazeném vektoru
- `dedup()` - po seřazení odstraní sousední duplicity

In [14]:
fn main() {
    let mut v: Vec<i32> = Vec::with_capacity(8);  // predalokovani pri inicializaci
    println!("{:?}", v);
    
    v.extend([5, 1, 2, 2, 4, 3]);
    println!("{:?}", v);

    println!("len={}, capacity={}, empty={}", v.len(), v.capacity(), v.is_empty());

    v.truncate(5);  // odstrani prvky od indexu n dale
    println!("{:?}", v);
    
    v.retain(|x| x % 2 == 0);  // ponecha jen sude prvky
    println!("{:?}", v);

    v.extend([4, 6, 6, 2]);
    println!("{:?}", v);
    
    v.sort();   // razeni vzestupne
    println!("{:?}", v);
    
    v.dedup();  // odstrani sousedni duplicity (po sort)
    println!("{:?}", v);
    
    v.sort_by(|a, b| b.cmp(a));  // razeni sestupne
    println!("{:?}", v);
    
    v.sort();
    println!("{:?}", v);
    
    match v.binary_search(&4) { // binary_search: vyzaduje vzestupne serazeny vektor
        Ok(i) => println!("4 nalezeno na indexu {i}, v={v:?}"),
        Err(i) => println!("4 nenalezeno, patrilo by na index {i}, v={v:?}"),
    }

    v.clear();  // vymazani vsech prvku
    println!("{:?}", v);
}
main();

[]
[5, 1, 2, 2, 4, 3]
len=6, capacity=8, empty=false
[5, 1, 2, 2, 4]
[2, 2, 4]
[2, 2, 4, 4, 6, 6, 2]
[2, 2, 2, 4, 4, 6, 6]
[2, 4, 6]
[6, 4, 2]
[2, 4, 6]
4 nalezeno na indexu 1, v=[2, 4, 6]
[]


### **PŘÍKLAD 3:** Vektory
Doplňte chybějící kód do funkcí pro práci s polem:
- `count_gt` - vrátí **počet** čísel větších než `threshold`
- `sort_and_dedup` - seřadí vektor a odstraní duplicity
- `sum_lt` - vrátí **součet** hodnot menších než `threshold`
- Do funkce `main` a definice názvů/parametrů/návratových hodnot funkcí **nezasahujte**!

In [15]:
fn count_gt(v: &Vec<i32>, threshold: i32) -> usize {
    v.iter().filter(|&&x| x > threshold).count()
}

fn sort_and_dedup(v: &mut Vec<i32>) {
    v.sort();
    v.dedup();
}

fn sum_lt(v: &Vec<i32>, threshold: i32) -> i32 {
    v.iter().filter(|&&x| x < threshold).sum()
}

fn main() {
    let mut v = vec![10, 5, 7, 5, 20, 3];

    println!("count_gt(6) = {}", count_gt(&v, 6));
    assert_eq!(count_gt(&v, 6), 3);

    println!("sum_lt(10) = {}", sum_lt(&v, 10));
    assert_eq!(sum_lt(&v, 10), 5 + 7 + 5 + 3);
    assert_eq!(sum_lt(&vec![10, 20], 10), 0);

    sort_and_dedup(&mut v);
    println!("sorted+dedup = {:?}", v);
    assert_eq!(v, vec![3, 5, 7, 10, 20]);
}

main();

count_gt(6) = 3
sum_lt(10) = 20
sorted+dedup = [3, 5, 7, 10, 20]


## Řezy (slices)
- **Slice** = řez = „pohled“ na souvislou část paměti *bez kopírování/přesunu*.
- Typy: `&[T]` (jen pro čtení), `&mut [T]` (čtení + zápis).
- Slice data *nevlastní* (jen si je zapůjčuje) a nese si jejich *délku*.
- Vzniká z pole / vektoru přes rozsahy: `&a[i..j]`.
- **POZOR:** Lze použít i u řetězce, ale může rozdělit vícebajtové UTF-8 znaky! (řeže po bajtech)

In [16]:
fn main() {
    let a = [10, 20, 30, 40, 50];

    let s1 = &a[1..4];  // 20,30,40
    let s2 = &a[..3];   // 10,20,30
    let s3 = &a[2..];   // 30,40,50
    let s4 = &a[1..=2]; // 20,30     ..= inkluzivni (1,2)
    let s5 = &a[..];    // cele pole jako slice
    
    println!("{:?} {:?} {:?} {:?} {:?}", s1, s2, s3, s4, s5);
    // [20, 30, 40] [10, 20, 30] [30, 40, 50] [20, 30] [10, 20, 30, 40, 50]
}
main();

[20, 30, 40] [10, 20, 30] [30, 40, 50] [20, 30] [10, 20, 30, 40, 50]


## Řetězce
- `String` = **řetězec** s daty na heapu, měnitelný (`push`, `insert`, ...)
- `&String` = **reference na řetězec**
- `&str` = **string slice** (pohled na řetězcová data uložená jinde)  
  Pozn.: literál `"ahoj"` je `&'static str`
- Pokud jen čteme, je lepší (pro univerzálnější použití) použít `&str` místo `&String`.  
  Očekáváme-li `&str` (např. argument funkce), převod proběhne automaticky.

In [17]:
fn greet(name: &str) { println!("Ahoj, {name}!"); }

fn main() {
    let s1: &str = "Ivo";
    let mut s2: String = String::from(s1); // &str -> String
    // let mut s2: String = s1.into();     // alternativa
    
    s2.push('!');
    greet(s1);
    greet(&s2);   // &String -> &str (automaticky prevod)
}
main();

Ahoj, Ivo!
Ahoj, Ivo!!


### Konkatenace a dekatenace řetězců
- Makro `format!`:
  - Spojí více hodnot do jednoho řetězce.
  - Vytvoří nový `String`, původní hodnoty si jen půjčí a hned vrátí.
  - Původní hodnoty se do nového řetězce **zkopírují**.
  - Pohodlnější, lze kombinovat i s čísly a dalšími typy.
- Operátor `+`:
  - Spojí `String + &str -> String`
  - Levá strana (`String`) se **přesune** (spotřebuje) a použije se jako "buffer", do kterého se připojí pravá část.
  - Pravá strana je **string slice** (`&str`), nikoli `String`.
  - Paměťově úspornější, nealokuje se nový řetězec.
- `msg.split(' ')`
  - Rozdělí řetězec podle znaku mezery a vrátí iterátor (`std::str::Split`) přes části typu `&str`.
  - Prvky lze sesbírat pomocí `collect()` do kolekce (např. vektoru).

In [18]:
fn main() {
    // ----------------------------------
    // Priklad spojovani pomoci 'format'
    // ----------------------------------
    let first = String::from("Jan");
    let second = String::from("Novak");
    let age = 20;
    
    let msg = format!("{} {} ma {} let.", first, second, age);

    println!("{msg}");    // Jan Novak ma 20 let.
    println!("{first}");  // Jan

    // --------------------------
    // Dekatenace pomoci 'split'
    // --------------------------
    for part in msg.split(' ') {  // Vraci iterator std::str::Split
        println!("{part}");  // Jan, Novak, ...
    }
    let parts: Vec<&str> = msg.split(' ').collect(); // do vektoru
    println!("{:?}", parts); // ["Jan","Novak","ma","20","let."]

    // -----------------------------
    // Priklad spojovani pomoci '+'
    // -----------------------------
    let a = String::from("Jan");
    let b = String::from("Novak");

    // '+' bere String vlevo a &str vpravo.
    // a se "spotrebuje" (move) a pouzije jako buffer pro vysledek.
    let full = a + " " + &b;

    println!("{full}");   // Jan Novak
    // println!("{a}");   // NEJDE: a bylo spotrebovano
    println!("{b}");      // Novak (b zustava pouzitelny)
}
main();

Jan Novak ma 20 let.
Jan
Jan
Novak
ma
20
let.
["Jan", "Novak", "ma", "20", "let."]
Jan Novak
Novak


### Užitečné metody pro práci s řetězci v Rustu

#### `&str` (string slice) - většina funguje i u `String`
- `len()` - délka **v bajtech** (pozor: ne počet znaků u UTF-8)
- `is_empty()` - je řetězec prázdný?
- `trim()` / `trim_start()` / `trim_end()` - ořeže bílé znaky
- `starts_with(...)` / `ends_with(...)` - test prefixu/sufixu
- `contains(...)` - obsahuje podřetězec/znak?
- `find(...)` / `rfind(...)` - najde pozici podřetězce (vrací `Option<usize>`)
- `split_whitespace()` - rozdělí podle mezer (iterátor slov)
- `split(sep)` / `splitn(n, sep)` - rozdělení podle oddělovače `sep`
- `lines()` - iterátor přes řádky
- `replace(from, to)` - nahradí všechny výskyty (vrací nový `String`)
- `to_lowercase()` / `to_uppercase()` - změní velikost písmen (vrací `String`)
- `chars()` - iterátor přes znaky (`char`)
- `bytes()` - iterátor přes bajty
- `parse::<T>()` - pokus o parsování do typu `T` (vrací `Result<T, _>`)

#### `String` (vlastněný řetězec)
- `new()` / `String::from(...)` / `"text".to_string()` - vytvoření `String`
- `push(ch)` - přidá znak na konec
- `push_str("...")` - přidá řetězec na konec
- `pop()` - odebere poslední znak (vrací `Option<char>`)
- `clear()` - vymaže obsah
- `insert(idx, ch)` / `insert_str(idx, "...")` - vloží na pozici (pozor na UTF-8 indexy)
- `remove(idx)` - odstraní znak na pozici (pozor na UTF-8 indexy)
- `truncate(n)` - zkrátí na `n` **bajtů** (pozor na UTF-8 hranice)
- `split_off(idx)` - rozdělí na dva `String` (od `idx` dál vrátí nový)
- `as_str()` - vrátí `&str` pohled na obsah
- `capacity()` / `reserve(n)` / `shrink_to_fit()` - práce s kapacitou
- `into_bytes()` - převede na `Vec<u8>`

#### Skládání/formátování řetězců
- `format!("...{x}...")` - vytvoří `String` (podobně jako `println!`, ale vrací řetězec)
- `join(sep)` - spojí kolekci řetězců separátorem (např. `Vec<&str>`)
- `collect::<String>()` - poskládá iterátor znaků do `String`

In [19]:
fn main() {
    // join
    let words = vec!["rust", "is", "fun"];
    let sentence = words.join(" ");
    println!("sentence: {sentence}");

    // split_whitespace
    let text = "a bb rust is cool ok";
    let mut long_words: Vec<&str> = Vec::new();
    for w in text.split_whitespace() {
        if w.len() > 2 {
            long_words.push(w);
        }
    }
    println!("long_words: {:?}", long_words);

    // replace
    let replaced = sentence.replace("fun", "fast");
    println!("replaced: {replaced}");

    // starts_with / ends_with / contains
    println!("starts_with 'rust': {}", replaced.starts_with("rust"));
    println!("contains 'fast': {}", replaced.contains("fast"));
    println!("ends_with 'fast': {}", replaced.ends_with("fast"));
}
main();

sentence: rust is fun
long_words: ["rust", "cool"]
replaced: rust is fast
starts_with 'rust': true
contains 'fast': true
ends_with 'fast': true


### **PŘÍKLAD 4:** Práce s řetězci
Doplňte chybějící kód do funkcí pro práci s řetězci:
- `normalize` - **Ořeže mezery** na začátku a konci a převede text na **malá písmena**.
- `acronym` - Vytvoří velkými písmeny **zkratku z prvních písmen** všech slov v řetězci.
- `longest_word_len` - Vrátí **délku nejdelšího slova** v řetězci.
- `vec_to_string` - Z vektoru `&str` postaví řetězec, přičemž jednotlivé části spojí mezerami.

In [22]:
fn normalize(raw: &str) -> String {
    raw.trim().to_lowercase()
}

fn acronym(parts: &[&str]) -> String {
    parts.iter().filter_map(|w| w.chars().next()).map(|c| c.to_ascii_uppercase()).collect()
}

fn longest_word_len(doc: &str) -> usize {
    doc.split_whitespace().map(|w| w.len()).max().unwrap_or(0)
}

fn vec_to_string(parts: &[&str]) -> String {
    parts.join(" ")
}

fn main() {
    let raw = "  HeLLo RuSt  ";
    let n = normalize(raw);
    println!("normalize = {n}");
    assert_eq!(n, "hello rust");

    let p = vec!["central", "processing", "unit"];
    let a = acronym(&p);
    println!("acronym = {a}");
    assert_eq!(a, "CPU");

    let doc = "rust makes systems programming fun";
    let m = longest_word_len(doc);
    println!("longest_word_len = {m}");
    assert_eq!(m, 11);

    let v = vec!["rust", "is", "fast"];
    let s = vec_to_string(&v);
    println!("vec_to_string = {s}");
    assert_eq!(s, "rust is fast");
}
main();

normalize = hello rust
acronym = CPU
longest_word_len = 11
vec_to_string = rust is fast
